In [1]:
#import logging
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.optim as optim
#from torch.utils.tensorboard import SummaryWriter
import torch
import unicodedata
import string
from tqdm import tqdm
import random

from typing import List
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from typing import Any, Dict
from pathlib import Path
import math
import h5py
import time
import re
import argparse
import timm
#from torch.utils.tensorboard import SummaryWriter

from aroma_Burgers import *
from DiT_Burgers import *
from metrics import *
from metrics_Burgers import StateMetrics,State_DiT,State

### TEST Metrics

In [ ]:
# For checkpoint
direc="/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/checkpoints/"
#direc = Path(cfg.save_dir)
#save_dir.mkdir(parents=True, exist_ok=True)
#ckpt_path = save_dir / "checkpoint.pt"

checkpoint_metrics=Path(direc+"checkpoint_metrics_encoDeco.pt")

checkpoint_los_tr_mean=Path(direc+"losses_train_metrics_encoDeco_m.npy")
checkpoint_los_tr_last=Path(direc+"losses_train_metrics_enoDeco_l.npy")
checkpoint_los_tr_attn=Path(direc+"losses_train_metrics_encoDeco_a.npy")

checkpoint_los_te_mean=Path(direc+"losses_test_metrics_encoDeco_m.npy")
checkpoint_los_te_last=Path(direc+"losses_test_metrics_encoDeco_l.npy")
checkpoint_los_te_attn=Path(direc+"losses_test_metrics_encoDeco_a.npy")


loss_train_metrics_mean=np.load(checkpoint_los_tr_mean)
loss_train_metrics_last=np.load(checkpoint_los_tr_last)
loss_train_metrics_attn=np.load(checkpoint_los_tr_attn)

loss_test_metrics_mean=np.load(checkpoint_los_te_mean)
loss_test_metrics_last=np.load(checkpoint_los_te_last)
loss_test_metrics_attn=np.load(checkpoint_los_te_attn)

### Affichage loss

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_train_metrics_mean, 
            color=colors[0], 
            label=f'train_mean',
            linewidth=2)

plt.plot(loss_train_metrics_last, 
            color=colors[1], 
            label=f'train_last',
            linewidth=2)

plt.plot(loss_train_metrics_attn, 
            color=colors[2], 
            label=f'train_attn',
            linewidth=2)

plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses train mean/attn/last', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_test_metrics_mean, 
            color=colors[0], 
            label=f'test_mean',
            linewidth=2)

plt.plot(loss_test_metrics_last, 
            color=colors[1], 
            label=f'test_last',
            linewidth=2)

plt.plot(loss_test_metrics_attn, 
            color=colors[2], 
            label=f'test_attn',
            linewidth=2)

plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses test mean/attn/last', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_train_metrics_attn, 
            color=colors[0], 
            label=f'train_attn',
            linewidth=2)

plt.plot(loss_test_metrics_attn, 
            color=colors[1], 
            label=f'test_attn',
            linewidth=2)


plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses test/test attn', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### TEST

In [ ]:
from aroma_Burgers import *

base = Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF")
fname = base / "dataset" /"CE_test_E1.h5"
with h5py.File(fname, "r") as f:
    data = f["test/pde_250-100"][:]

x_dim=1
batch_size=1
test_data=Dataset_pde(data,x_dim,x_min=0,x_max=16,t_min=0,t_max=4,forescast=False)

#kwargs = {"num_workers": 4, "pin_memory": True} if cfg.use_gpu else {}
test_loader=DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)
test_loader_cycle=cycle(test_loader)

### TEST FOR DIT-DPO

In [ ]:
direc="/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/checkpoints/"


checkpoint_los_tr_DDD=Path(direc+"losses_train_DiT_DPO.npy")
#checkpoint_los_tr_DD=Path(direc+"losses_train.npy")


checkpoint_los_te_DDD=Path(direc+"losses_test_DiT_DPO.npy")
#checkpoint_los_te_ED=Path(direc+"losses_test.npy")


loss_train_DDD=np.load(checkpoint_los_tr_DDD)
#loss_train_ED=np.load(checkpoint_los_tr_ED)

loss_test_DDD=np.load(checkpoint_los_te_DDD)
#loss_test_ED=np.load(checkpoint_los_te_ED)




In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_train_DDD, 
            color=colors[0], 
            label=f'train_loss',
            linewidth=2)

plt.plot(loss_test_DDD, 
            color=colors[1], 
            label=f'test_loss',
            linewidth=2)


plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses test/train DPO', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
checkpoint_f = Path(direc+"checkpoint.pt")
checkpoint_DiT_DPO = Path(direc+"checkpoint_DiT_DPO.pt")
checkpoint_DiT=Path(direc+"checkpoint_DiT.pt")
checkpoint_encoDecoDPO=Path(direc+"checkpoint_enco_deco_DPO.pt")
#checkpoint_metrics_lastModel=Path(direc+"checkpoint_metrics_lastModel.pt")

print("resolve==============",checkpoint_f.resolve())


if checkpoint_encoDecoDPO.exists() and checkpoint_encoDecoDPO.is_file():
    with checkpoint_encoDecoDPO.open("rb") as fp:
        state_enco_decoDPO=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco DPO reussi----------------------")

if checkpoint_DiT_DPO.exists() and checkpoint_DiT_DPO.is_file():
    with checkpoint_DiT_DPO.open("rb") as fp:
        state_DiT_DPO=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state DiT DPO reussi----------------------")

if checkpoint_f.exists() and checkpoint_f.is_file():
    with checkpoint_f.open("rb") as fp:
        state_enco_deco=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco reussi----------------------")

if checkpoint_DiT.exists() and checkpoint_DiT.is_file():
    with checkpoint_DiT.open("rb") as fp:
        state_DiT=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state DiT reussi----------------------")


In [ ]:
num_sample=5
eps=1e-32
device=torch.device("cpu")

pertu=0.5
dx=0.16
mass_weight=10.
energy_weight=1
grad_weight=30.
boundary_weight=0.1

min_noise_std = 1e-2 #cfg.min_noise_std  # 2e-6
num_refinement_steps=3 #cfg.num_refinement_steps
betas = [
    min_noise_std ** (k / num_refinement_steps)
    for k in reversed(range(num_refinement_steps + 1))
]

scheduler = DDPMScheduler(
    num_train_timesteps=num_refinement_steps + 1,
    trained_betas=betas,
    prediction_type="v_prediction",
    clip_sample=False,
)
time_multiplier = 1000 / num_refinement_steps

with torch.no_grad():
                
             

    batch=next(test_loader_cycle)

    state_enco_decoDPO.model_enco.eval()
    state_enco_decoDPO.model_deco.eval()

    state_enco_deco.model_enco.eval()
    state_enco_deco.model_deco.eval()

    state_DiT_DPO.model.eval()
    state_DiT.model.eval()

    
   

    x=batch[0]
    u=batch[1]

    x=x.to(device)
    u=u.to(device)

    x_out=x[:,1:,:,:]
    target=u[:,1:,:,:] #batch (b T-1 N d)

    batch_size_test,time_step_test,space_step,_=x.shape

             
    u=rearrange(u,"b T N d -> (b T) N d")  #shape (E, b, T, N, d)

    #x=x.expand(num_sample, *x.shape)            #shape (E, b, T, N, d)
    x=rearrange(x,"b T N d -> (b T) N d")
        

    sampleDPO,_,_,_=state_enco_decoDPO.model_enco(x,u) #shape (( E batch,T),M,h)
    sample,_,_,_=state_enco_deco.model_enco(x,u) #shape (( E batch,T),M,h)


    
    sammple_amenaged=rearrange(sample,"(b T) N d -> b T N d",b=batch_size_test)
    sammple_amenagedDPO=rearrange(sampleDPO,"(b T) N d -> b T N d",b=batch_size_test)

    y=[]  
    yDPO=[]  
    for t in range(time_step_test - 1 ):
        
        sample_prev=sammple_amenaged[:,t,:,:]  #sample_prev shape (b,N,d)
        sample_cur=sammple_amenaged[:,t+1,:,:] #sample_cur

        sample_prevDPO=sammple_amenagedDPO[:,t,:,:]  #sample_prev shape (b,N,d)
        sample_curDPO=sammple_amenagedDPO[:,t+1,:,:] #sample_cur


        y_noisedDPO = torch.randn_like(sample_curDPO)  # , dtype=sample_cur.dtype, device=sample_cur.device
        y_noised = torch.randn_like(sample_cur)  # , dtype=sample_cur.dtype, device=sample_cur.device


        for k in scheduler.timesteps:
            timess = (
                torch.zeros(
                    size=(sample_cur.shape[0],), dtype=sample_cur.dtype, device=sample_cur.device
                )
                + k
            )
            predDPO = state_DiT_DPO.model(
                torch.cat([sample_prevDPO, y_noisedDPO], dim=1), timess * time_multiplier
            )
            y_noisedDPO = scheduler.step(predDPO, k, y_noisedDPO).prev_sample # shape (Eb,M,h)


            pred = state_DiT.model(
                torch.cat([sample_prev, y_noised], dim=1), timess * time_multiplier
            )
            y_noised = scheduler.step(pred, k, y_noised).prev_sample # shape (Eb,M,h)
            
        yDPO.append( y_noisedDPO.unsqueeze(1)) 
        y.append( y_noised.unsqueeze(1)) 
        

    
    
    y=torch.cat(y,dim=1)  # # shape (b,T-1,M,h)
    yDPO=torch.cat(yDPO,dim=1)  # # shape (b,T-1,M,h)
    
    y=rearrange(y, "b T M h -> (b T) M h")
    yDPO=rearrange(yDPO, "b T M h -> (b T) M h")


    #x_out=x_out.expand(num_sample, *x_out.shape)
    x_out=rearrange(x_out,"b T M h -> (b T) M h")

    outDiT=state_enco_deco.model_deco(y,x_out) # shape (Eb(T-1),N,d)
    outDiT=rearrange(outDiT, "(b T) M h ->b T M h",b=batch_size_test) # shape (E, b, (T-1),N,d)

    outDiTDPO=state_enco_decoDPO.model_deco(yDPO,x_out) # shape (Eb(T-1),N,d)
    outDiTDPO=rearrange(outDiTDPO, "(b T) M h ->b T M h",b=batch_size_test) # shape (E, b, (T-1),N,d)

        
        

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

tims=248
space=x[0].squeeze(-1)
space=space.detach().numpy()

outDiTDPO=outDiTDPO.detach().numpy()
outDiT=outDiT.detach().numpy()
target=target.detach().numpy()


plt.plot(space , outDiTDPO[0,tims,:,0],
            label=f'DiT_DPO',
            linewidth=2)

plt.plot(space , outDiT[0,tims,:,0],
            label=f'DiT',
            linewidth=2)

plt.plot(space, target[0,tims,:,0],
            label=f'target',
            linewidth=2)


plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title(f'Burgers equation t={tims}', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Tes données
# x : (Nx,)
# t : (Nt,)
# sol1, sol2, sol3 : (Nt, Nx)

plt.style.use("seaborn-v0_8-darkgrid")

fig, ax = plt.subplots(figsize=(8, 5))

# Courbes
line1, = ax.plot([], [], lw=2, label="DiT")
line2, = ax.plot([], [], lw=2, label="DiT-DPO")
line3, = ax.plot([], [], lw=2, label="Target")

len_t=248
time_st=np.linspace(0,4,250)

# Limites fixes (important pour animation propre)
ax.set_xlim(space.min(), space.max())
ax.set_ylim(
    min(outDiT.min(), outDiT.min(), target.min()),
    max(outDiT.max(), outDiTDPO.max(), target.max())
)

ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("u(x,t)", fontsize=12)
title = ax.set_title("", fontsize=14)

ax.legend(loc="upper right")

# Initialisation
def init():
    line1.set_data([], [])
    line2.set_data([], [])
    line3.set_data([], [])
    title.set_text("")
    return line1, line2, line3, title

# Update à chaque frame
def update(frame):
    line1.set_data(space, outDiT[0,frame,:,0])
    line2.set_data(space, outDiTDPO[0,frame,:,0])
    line3.set_data(space, target[0,frame,:,0])

    title.set_text(f"Équation de Burgers  —  t = {time_st[frame]:.2f}")

    return line1, line2, line3, title

# Animation
ani = FuncAnimation(
    fig,
    update,
    frames=len_t,
    init_func=init,
    blit=True,
    interval=80   # vitesse (plus petit = plus rapide)
)
ani.save("burgers4.gif", writer="pillow", fps=10)
plt.tight_layout()
plt.show()

### TEST Encoder-Decoder Metrics

In [ ]:
checkpoint_f = Path(direc+"checkpoint.pt")
checkpoint_metrics=Path(direc+"checkpoint_metrics_encoDeco.pt")
checkpoint_metrics_lastModel=Path(direc+"checkpoint_metrics_encoDecoLastModel.pt")

#checkpoint_nan = Path(direc+"/checkpoint_nan.pt")
print("resolve==============",checkpoint_f.resolve())


if checkpoint_f.exists() and checkpoint_f.is_file():
    with checkpoint_f.open("rb") as fp:
        state_enco_deco=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco reussi----------------------")


if checkpoint_metrics.exists() and checkpoint_metrics.is_file():
    with checkpoint_metrics.open("rb") as fp:
        state_metrics=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state Metrics enco deco reussi----------------------")

if checkpoint_metrics_lastModel.exists() and checkpoint_metrics_lastModel.is_file():
    with checkpoint_metrics.open("rb") as fp:
        state_metrics_lastModel=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state Metrics enco deco last model reussi----------------------")

In [ ]:
num_sample=5
eps=1e-32
device=torch.device("cpu")

pertu=0.4

sigmoid=nn.Sigmoid()

batch=next(test_loader_cycle)
state_enco_deco.model_enco.eval()
state_enco_deco.model_deco.eval()

state_metrics.model_mean.eval()
state_metrics.model_last.eval()
state_metrics.model_attn.eval()

x=batch[0]
u=batch[1]

x=x.to(device)
u=u.to(device)

x_out=x
target=u #batch (b T N d)

batch_size_test,time_step_test,space_step,_=x.shape

pertubation=pertu*torch.randn( num_sample,*u.shape,device=device) #shape (E, b, T, N, d)

u=u.unsqueeze(0)+ pertubation              #shape (E, b, T, N, d)
u=rearrange(u,"E b T N d -> (E b T) N d")

x=x.expand(num_sample, *x.shape)            #shape (E, b, T, N, d)
x=rearrange(x,"E b T N d -> (E b T) N d")

with autocast(enabled=False): # mixed precision
    #state_enco_deco.model_enco.eval()
    #samples=[]
    

    sample,_,_,_=state_enco_deco.model_enco(x,u) #shape ((E, batch,T),M,h)

    x_out=x_out.expand(num_sample, *x_out.shape)
    x_out=rearrange(x_out,"E b T M h -> (E b T) M h")

    out=state_enco_deco.model_deco(sample,x_out) # shape (Eb(T),N,d)
    out=rearrange(out, "(E b T) M h ->E b T M h",E=num_sample,b=batch_size_test) # shape (E, b, (T),N,d)

    


In [ ]:
dx=0.16
mass_weight=100.
energy_weight=1.
grad_weight=10.
boundary_weight=0.1

winner,loser=winLos(out,target.unsqueeze(0),dx,mass_weight,energy_weight,grad_weight,boundary_weight) # shape ( b, T,N,d)
        
winner_score_m=state_metrics.model_mean(winner) #(b,T)
winner_score_l=state_metrics.model_last(winner)
winner_score_a=state_metrics.model_attn(winner)

loser_score_m=state_metrics.model_mean(loser)
loser_score_l=state_metrics.model_last(loser)
loser_score_a=state_metrics.model_attn(loser)

loss_m=-torch.log(eps+ sigmoid(winner_score_m-loser_score_m))
#loss_m=loss_m.mean()

loss_l=-torch.log(eps+ sigmoid(winner_score_l-loser_score_l))
#loss_l=loss_l.mean()

loss_a=-torch.log(eps+ sigmoid(winner_score_a-loser_score_a))
#loss_a=loss_a.mean()


In [ ]:
mask_m=loss_m<=-np.log(0.5)
mask_a=loss_a<=-np.log(0.5)
mask_l=loss_l<=-np.log(0.5)
print(torch.sum(mask_m,dim=1))
print(torch.sum(mask_a,dim=1))
print(torch.sum(mask_l,dim=1))


In [ ]:
inddd=0
w=target[0,0]
l=out[inddd,0,0]
w=w.unsqueeze(0)
w=w.unsqueeze(0)

l=l.unsqueeze(0)
l=l.unsqueeze(0)

w_score_a=state_metrics.model_attn(w)
l_score_a=state_metrics.model_attn(l)
print(w_score_a)
print(l_score_a)


In [ ]:
x_space=np.linspace(0,16,100)

plt.plot(x_space, w[0,0].detach().numpy(), 
            label=f'sample winer ',
            linewidth=2)

   
plt.plot(x_space, l[0,0].detach().numpy(), 
            label=f'sample loser',
            linewidth=2)

plt.title('Losses test/test mean', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
ind=200
time_indices = [0, 1]
x_space=np.linspace(0,16,100)
#colors = plt.cm.viridis(np.linspace(0, 1, num_sample+3))
# Plot à différents instants
plt.figure(figsize=(12, 6))

for i in range(num_sample):
    plt.plot(x_space, out[i,0,ind,:,0], 
            label=f'{i}',
            linewidth=2)

plt.plot(x_space, winner[0,ind,:,0], 
            label=f'sample winer ',
            linewidth=2)

   
plt.plot(x_space, loser[0,ind,:,0], 
            label=f'sample loser',
            linewidth=2)

plt.plot(x_space, target[0,ind,:,0], 
            label=f'target',
            linewidth=2)





plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses test/test mean', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### TEST ENCODER DECODER DPO

In [ ]:
# For checkpoint
direc="/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/checkpoints/"

checkpoint_los_tr_EDD=Path(direc+"losses_train_enco_deco_DPO.npy")
checkpoint_los_tr_ED=Path(direc+"losses_train.npy")


checkpoint_los_te_EDD=Path(direc+"losses_test_enco_deco_DPO.npy")
checkpoint_los_te_ED=Path(direc+"losses_test.npy")


loss_train_EDD=np.load(checkpoint_los_tr_EDD)
loss_train_ED=np.load(checkpoint_los_tr_ED)

loss_test_EDD=np.load(checkpoint_los_te_EDD)
loss_test_ED=np.load(checkpoint_los_te_ED)


In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_train_EDD[-50:], 
            color=colors[0], 
            label=f'train_DPO',
            linewidth=2)

plt.plot(loss_train_ED[-50:], 
            color=colors[1], 
            label=f'train_no_DPO',
            linewidth=2)


plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses train over the last 50 epochs', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from aroma_Burgers import *

base = Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF")
fname = base / "dataset" /"CE_test_E1.h5"
with h5py.File(fname, "r") as f:
    data = f["test/pde_250-100"][:]

x_dim=1
batch_size=1
test_data=Dataset_pde(data,x_dim,x_min=0,x_max=16,t_min=0,t_max=4,forescast=False)

#kwargs = {"num_workers": 4, "pin_memory": True} if cfg.use_gpu else {}
test_loader=DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)
test_loader_cycle=cycle(test_loader)

In [ ]:
checkpoint_f = Path(direc+"checkpoint.pt")
checkpoint_metrics=Path(direc+"checkpoint_metrics_encoDeco.pt")
checkpoint_encoDecoDPO=Path(direc+"checkpoint_enco_deco_DPO.pt")
checkpoint_encoDecoDPO_last=Path(direc+"checkpoint_enco_deco_DPO_lastModel.pt")


#checkpoint_nan = Path(direc+"/checkpoint_nan.pt")
print("resolve==============",checkpoint_f.resolve())


if checkpoint_f.exists() and checkpoint_f.is_file():
    with checkpoint_f.open("rb") as fp:
        state_enco_deco=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco reussi----------------------")

if checkpoint_encoDecoDPO.exists() and checkpoint_encoDecoDPO.is_file():
    with checkpoint_encoDecoDPO.open("rb") as fp:
        state_enco_decoDPO=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco DPO reussi----------------------")

if checkpoint_metrics.exists() and checkpoint_metrics.is_file():
    with checkpoint_metrics.open("rb") as fp:
        state_metrics=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state Metrics reussi----------------------")

if checkpoint_encoDecoDPO_last.exists() and checkpoint_encoDecoDPO_last.is_file():
    with checkpoint_encoDecoDPO_last.open("rb") as fp:
        state_encoDeco_lastModel=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state encoDecoDPO last model reussi----------------------")

In [ ]:

device=torch.device("cpu")


batch=next(test_loader_cycle)
state_enco_deco.model_enco.eval()
state_enco_deco.model_deco.eval()

state_enco_decoDPO.model_enco.eval()
state_enco_decoDPO.model_deco.eval()

with torch.no_grad():

    x=batch[0]
    u=batch[1]

    x=x.to(device)
    u=u.to(device)

    #x_out=x
    target=u #batch (b T N d)

    batch_size_test,time_step_test,space_step,_=x.shape
    u=rearrange(u,"b T N d -> (b T) N d")

    x=x.expand(*x.shape)            #shape (E, b, T, N, d)
    x=rearrange(x,"b T N d -> (b T) N d")


    sample,_,_,_=state_enco_deco.model_enco(x,u) #shape ((batch,T),M,h)
    sampleDPO,_,_,_=state_enco_decoDPO.model_enco(x,u)

    out=state_enco_deco.model_deco(sample,x) # shape (Eb(T),N,d)
    out=rearrange(out, "(b T) M h -> b T M h",b=batch_size_test) # shape ( b, (T),N,d)

    outDPO=state_enco_decoDPO.model_deco(sampleDPO,x)
    outDPO=rearrange(outDPO,"(b T) M h -> b T M h",b=batch_size_test)

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

tims=0
space=x[0].squeeze(-1)
plt.plot(space.detach().numpy() , out[0,tims,:,0].detach().numpy(),
            label=f'no_DPO',
            linewidth=2)

plt.plot(space.detach().numpy() , outDPO[0,tims,:,0].detach().numpy(),
            label=f'DPO',
            linewidth=2)
plt.plot(space.detach().numpy() , target[0,tims,:,0].detach().numpy(),
            label=f'target',
            linewidth=2)


plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title(f'Burgers equation t={tims}', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### SOMES TESTS

In [ ]:
dataa=next(test_loader_cycle)
u=dataa[1]
x=dataa[0]

dx=x[0,0,1,0]-x[0,0,0,0]
metric=PhysicalMetricsBurgers(dx=dx.item())
ener=metric.energy(u) #shape (batch,T)
mass_c=metric.mass(u) #shape shape (batch,T)
gred_x=metric.gradient(u) # # shape (batch,T,N-2)
bound=metric.boundary_condition(u) #shape shape (batch,T)

depth=4
input_size=1
output_size=1
num_heads=8
hidden_size=16*num_heads
frequency_embedding_size=256
num_point=100
#u_embeder=nn.Linear(input_size,hidden_size)
balance=RewardSignal(depth,input_size,num_heads,hidden_size,output_size=output_size,num_point=num_point, frequency_embedding_size=frequency_embedding_size,reduction="attention",x_space="regular")





In [ ]:
num_refinement_steps =3

min_noise_std =   2e-6
betas = [
    min_noise_std ** (k / num_refinement_steps)
    for k in reversed(range(num_refinement_steps + 1))
]

scheduler = DDPMScheduler(
        num_train_timesteps=num_refinement_steps + 1,
        trained_betas=betas,
        prediction_type="v_prediction",
        clip_sample=False,
    )
time_multiplier = 1000 / num_refinement_steps

k=3
noise_factor = scheduler.alphas_cumprod
#noise_factor = noise_factor
#noise_factor = noise_factor.view(-1, *[1 for _ in range(sample_prev.ndim - 1)]) #shape: (batch,1,1)
signal_factor = 1 - noise_factor
print("betas", betas)
print("noise_factor",noise_factor)

k=torch.tensor(k)
sample_cur=torch.ones(2,3)
noise=torch.randn_like(sample_cur)

sample_noised =scheduler.add_noise(sample_cur, noise, k)
print("sample_cur",sample_cur)
print("sample_noised",sample_noised)
print("noise",noise)
print("---------------------------------------------------")
y_noised = scheduler.step(noise, k, noise).prev_sample
print("y_noised", y_noised)
check=(noise_factor[3]**0.5 -signal_factor[3]**0.5)*noise
print("test",check )
print(torch.allclose(check,y_noised))

In [ ]:
signal_factor[3]

In [ ]:
print(x.shape)
print(u.shape)
print(16/100)
print(f"difference, {x[0,0,1]-x[0,0,0]}, initit {x[0,0,0]} final {x[0,0,-1]}")
print(f"difference: {x[0,0,1:,0]-x[0,0,:-1,0]}")
print("len", len(x[0,0,1:,0]-x[0,0,:-1,0]))

In [ ]:
# Example of use
t_emb = SinusoidalPositionEmbeddings(10)
timess=torch.arange(13)
t_emb(timess).shape
timess

In [ ]:
Y_pred=torch.rand(10,32,10,50,1)
Y_true=torch.rand(1,32,10,50,1)
dx=0.16
mass_weight=1.
energy_weight=0.1
grad_weight=10
boundary_weight=0.1
winn,los=winLos(Y_pred,Y_true,dx,mass_weight,energy_weight,grad_weight,boundary_weight)

In [ ]:
m=physicalMetricsBurgers(Y_pred,Y_true,dx,mass_weight,energy_weight,grad_weight,boundary_weight)
print(winn.shape)
print(los.shape)
print(m.shape)

In [ ]:
import torch.nn.functional as F

A=torch.arange(12).view(3,4)

pd1=(0,1,0,1)
out=F.pad(A.unsqueeze(0),pd1,"circular")
out=out.squeeze(0)
print("A",A)
print("out",\
      out)

### FFT

In [2]:
from scipy.fft import fft, fftfreq, ifft
N=10
L=1
T=L/N
x=np.linspace(0,1,10)
f=np.cos(x)
f_p=-np.sin(x)
fp_hat=fft(f_p)
f_hat=fft(f)

freq=2*np.pi*1j*fftfreq(N,T)
fp_hat2=freq*f_hat


print(fp_hat)
print(fp_hat2)

[-4.5537574 -0.j          0.61498534-1.42511226j  0.49313574-0.62230496j
  0.47177026-0.32710178j  0.4652045 -0.14609324j  0.46356574-0.j
  0.4652045 +0.14609324j  0.47177026+0.32710178j  0.49313574+0.62230496j
  0.61498534+1.42511226j]
[ 0.        +0.j          5.55857371+0.05860314j  4.85453408+2.58381093j
  3.82752868+4.52446935j  2.27931537+6.29844739j -0.        -7.95599287j
  2.27931537-6.29844739j  3.82752868-4.52446935j  4.85453408-2.58381093j
  5.55857371-0.05860314j]


In [8]:
from torch.fft import fft2,ifft2,fftfreq
Lx=4
Lz=2
Nx=100
Nz=50

dx=Lx/Nx
dz=Lz/Nz

kx=2*torch.pi*fftfreq(Nx,d=dx)
kz=2*torch.pi*fftfreq(Nz,d=dz)


kxx,kzz=torch.meshgrid(kx,kz,indexing='ij')
print(kxx.shape)

k2=torch.sqrt(kxx**2 +kzz**2)
print(k2.shape)
print(kxx)

torch.Size([100, 50])
torch.Size([100, 50])
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 1.5708,  1.5708,  1.5708,  ...,  1.5708,  1.5708,  1.5708],
        [ 3.1416,  3.1416,  3.1416,  ...,  3.1416,  3.1416,  3.1416],
        ...,
        [-4.7124, -4.7124, -4.7124,  ..., -4.7124, -4.7124, -4.7124],
        [-3.1416, -3.1416, -3.1416,  ..., -3.1416, -3.1416, -3.1416],
        [-1.5708, -1.5708, -1.5708,  ..., -1.5708, -1.5708, -1.5708]])
